# 04. 데이터 전처리 및 정규화

MAP API에서 수집한 상품 메타데이터를 정규화하고 CSV 테이블로 변환합니다.

In [ ]:
# Papermill parameters
env = "stg"
dt = "2025-08-11"
version_date = "20250811"
gcs_bucket_name = "air-airflow-stg"

In [ ]:
import os
import sys
import pandas as pd
import json
import ast
from typing import List, Dict, Any
from google.cloud import storage

In [ ]:
# GCS 경로 설정
gcs_client = storage.Client()
bucket = gcs_client.bucket(gcs_bucket_name)

# 입력 파일 경로 (02_collect_product_details에서 생성한 파일)
input_blob_path = f"product_meta_raw/{dt}/mobile_plan_info_{version_date}.json"

# 출력 파일 경로
output_prefix = f"preprocessing_result/{dt}/"

print(f"Processing data from: gs://{gcs_bucket_name}/{input_blob_path}")
print(f"Results will be saved to: gs://{gcs_bucket_name}/{output_prefix}")

In [ ]:
# pandas 옵션 설정
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)

In [ ]:
from config import DROP_COLUMNS, TABLE_COLUMNS, FIELD_EXTRACTIONS

In [ ]:
from utils import (
    extract_sub_field, str_to_num, to_gb, to_mbps, remove_won, safe_literal_eval,
    parse_offer_benefits, parse_data_option_providing_method,
    create_table_from_config, merge_product_lists, process_relation_data, remove_duplicates_keep_concurrent
)

In [ ]:
def save_to_gcs(df: pd.DataFrame, filename: str, prefix: str = output_prefix) -> None:
    """데이터프레임을 GCS에 CSV로 저장"""
    blob_path = f"{prefix}{filename}"
    blob = bucket.blob(blob_path)
    
    # CSV를 문자열로 변환하고 업로드
    csv_string = df.to_csv(index=False, encoding='utf-8-sig')
    blob.upload_from_string(csv_string, content_type='text/csv')
    
    print(f"Saved {filename} to gs://{gcs_bucket_name}/{blob_path}")

## 1. 데이터 로드

In [ ]:
# GCS에서 데이터 로드
print(f"Loading data from gs://{gcs_bucket_name}/{input_blob_path}")

try:
    blob = bucket.blob(input_blob_path)
    json_content = blob.download_as_text()
    data = json.loads(json_content)
    
    meta_data = pd.json_normalize(data)
    print(f"Data loaded successfully. Shape: {meta_data.shape}")
    print(f"Sample data:")
    display(meta_data.head(1))
    
except Exception as e:
    print(f"Error loading data: {e}")
    raise

## 2. 불필요한 컬럼 제거

In [ ]:
# 필요 없는 컬럼 제거
existing_drop_cols = [col for col in DROP_COLUMNS if col in meta_data.columns]
print(f"Dropping {len(existing_drop_cols)} columns")

try:
    meta_data.drop(columns=existing_drop_cols, inplace=True)
except KeyError as e:
    print(f"Warning: Some columns not found - {e}")

print(f"Remaining columns: {meta_data.shape[1]}")

## 3. 컬럼명 변환을 위한 매핑 생성

In [ ]:
token_set = set()
dup_set = set()

for field in meta_data.columns:
    field = field.replace('|', '.')
    field_parts = field.split('.')
    if field_parts[-1] in token_set:
        dup_set.add(field_parts[-1])
    token_set.add(field_parts[-1])

# 컬럼 매퍼 생성
column_mapper = {}
for field in meta_data.columns:
    field = field.replace('|', '.')
    field_parts = field.split('.')
    top_field = field_parts[0]
    
    if field_parts[-1] in ['value', 'valueList']:
        if field_parts[-2] in dup_set:
            column_mapper[field] = f"{top_field}.{field_parts[-2]}"
        else:
            if len(field_parts) > 1:
                column_mapper[field] = f"{top_field}.{field_parts[-2]}"
            else:
                column_mapper[field] = field_parts[-2]
    else:
        if field_parts[-1] in dup_set:
            column_mapper[field] = f"{top_field}.{field_parts[-2:]}"
        else:
            if len(field_parts) > 1:
                column_mapper[field] = f"{top_field}.{field_parts[-1]}"
            else:
                column_mapper[field] = field_parts[-1]

# 매핑 정보를 GCS에 저장
mapper_json = json.dumps(column_mapper, indent=4)
blob = bucket.blob(f"{output_prefix}column_mapper.json")
blob.upload_from_string(mapper_json, content_type='application/json')
print(f"Column mapper saved to GCS")

## 4. 중첩 필드 추출

In [ ]:
# value/valueList가 포함된 중첩 필드 단순화
print("Extracting nested fields...")

for source_field, sub_fields in FIELD_EXTRACTIONS.items():
    if source_field in meta_data.columns:
        print(f"Processing {source_field}")
        
        for sub_field in sub_fields:
            new_col_name = f"{source_field}.{sub_field}"
            meta_data[new_col_name] = meta_data[source_field].apply(
                extract_sub_field, args=(sub_field,)
            )
        
        # 원래 컬럼 삭제
        meta_data.drop(columns=[source_field], inplace=True)

## 5. 음성통화 필드 병합

In [ ]:
# 음성통화 제공량 관련 필드 정리
print("Merging voice fields...")
providing_amount_col = 'voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.providingAmount'
target_col = 'voice.includedVoiceCall.value'

if providing_amount_col in meta_data.columns and target_col in meta_data.columns:
    mask = ~meta_data[providing_amount_col].isnull()
    meta_data.loc[mask, target_col] = meta_data.loc[mask, providing_amount_col]

    # 불필요한 컬럼 삭제
    drop_cols = [
        'voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.providingAmount',
        'voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.range',
        'voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.providingAmount',
        'voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.range'
    ]
    existing_drop_cols = [col for col in drop_cols if col in meta_data.columns]
    meta_data.drop(columns=existing_drop_cols, inplace=True)

## 6. 혜택 및 데이터 옵션 정제

In [ ]:
# 요금제 연결 혜택/옵션 정제
print("Processing benefit fields...")

# allOfferBenefits 처리
all_offer_col = 'productBenefitConditions.allOfferBenefits'
if all_offer_col in meta_data.columns:
    meta_data[f'{all_offer_col}.benefitInfo'] = meta_data[all_offer_col].apply(
        parse_offer_benefits
    )
    meta_data.drop(columns=[all_offer_col], inplace=True)

# optionalOfferBenefits 처리
optional_offer_col = 'productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList'
if optional_offer_col in meta_data.columns:
    meta_data[optional_offer_col] = meta_data[optional_offer_col].apply(
        parse_offer_benefits
    )

# dataOptionProvidingMethod 처리
data_option_col = 'optionData.dataOptionProvidingMethod'
if data_option_col in meta_data.columns:
    meta_data[data_option_col] = meta_data[data_option_col].apply(
        parse_data_option_providing_method
    )

In [ ]:
# Step 1 저장
save_to_gcs(meta_data, 'step_1.csv')
print("Step 1 completed")

## 7. 숫자 변환

In [ ]:
# 음성/SMS/데이터/속도/가격 필드를 숫자형으로 변환
print("Applying numeric conversions...")

# 음성/SMS 필드 변환
voice_fields = [
    'voice.includedVoiceCall.value',
    'voice.includedVideoOrValueAddedCall.value',
    'smsText.includedText.value'
]
for field in voice_fields:
    if field in meta_data.columns:
        meta_data[field] = meta_data[field].apply(str_to_num)

# 데이터 필드 변환
data_fields = [
    'includedData.value',
    'additionalDataUsage.includedDataForSharingAndTethering.value',
    'additionalDataUsage.includedMVoIP.value',
    'benefitOfData.dataOptionRefill.dataRefillAmount.value'
]
for field in data_fields:
    if field in meta_data.columns:
        if field in ['additionalDataUsage.includedDataForSharingAndTethering.value',
                    'additionalDataUsage.includedMVoIP.value']:
            meta_data[field] = meta_data[field].fillna('0GB')
        elif field == 'benefitOfData.dataOptionRefill.dataRefillAmount.value':
            meta_data[field] = meta_data[field].fillna('무제한')

        meta_data[field] = meta_data[field].apply(to_gb)

# 속도 필드 변환
speed_field = 'dataQoS.appliedSpeed.value'
if speed_field in meta_data.columns:
    meta_data[speed_field] = meta_data[speed_field].fillna('0Mbps')
    meta_data[speed_field] = meta_data[speed_field].apply(to_mbps)

# 가격 필드 변환
price_fields = [
    'monthlyPrice.monthlyPrice.value',
    'monthlyPrice.monthlyPriceWithoutVAT.value',
    'monthlyPrice.monthlyPriceWithSelectableInstallment.value'
]
for field in price_fields:
    if field in meta_data.columns:
        meta_data[field] = meta_data[field].apply(remove_won)

# netPrice 특수 처리 (monthlyPrice * 0.9901)
net_price_field = 'salesInfo.netPrice.value'
monthly_price_field = 'monthlyPrice.monthlyPrice.value'
if net_price_field in meta_data.columns and monthly_price_field in meta_data.columns:
    mask = meta_data[net_price_field].isnull()
    if mask.any():
        calculated_price = (meta_data.loc[mask, monthly_price_field] * 0.9901).round().astype(int)
        meta_data.loc[mask, net_price_field] = calculated_price.astype(str) + '원'

# netPrice를 숫자로 변환
if net_price_field in meta_data.columns:
    meta_data[net_price_field] = meta_data[net_price_field].apply(remove_won)

In [ ]:
# Step 2 저장
save_to_gcs(meta_data, 'step_2.csv')
print("Step 2 completed")

## 8. 그룹 정보 정제

In [ ]:
# 상품 그룹 정보 처리
print("Processing product groups...")

group_col = 'otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList'
if group_col not in meta_data.columns:
    print(f"Warning: {group_col} not found in data")
else:
    # 그룹 정보 처리
    tmp_df = pd.DataFrame(meta_data[group_col].value_counts())
    tmp_df.reset_index(inplace=True)
    tmp_df.columns = ['group_list', 'count'] 
    tmp_df['group_list'] = tmp_df['group_list'].apply(ast.literal_eval) 
    tmp_df = tmp_df.explode(column='group_list')
    tmp_df.drop(columns='count', inplace=True)
    tmp_df = tmp_df.drop_duplicates()

    # 그룹 정보 추출
    tmp_df['groupName'] = tmp_df['group_list'].apply(lambda x: x.get('groupName') if isinstance(x, dict) else None)
    tmp_df['groupList'] = tmp_df['group_list'].apply(lambda x: x.get('groupList') if isinstance(x, dict) else None)
    tmp_df.drop(columns='group_list', inplace=True)

    # 제품 그룹 테이블 생성
    product_group = tmp_df.explode(column='groupList').reset_index(drop=True)
    product_group['pmProductId'] = product_group['groupList'].apply(
        lambda x: x.get('pmProductId') if isinstance(x, dict) else None
    )
    product_group['legacyProductId'] = product_group['groupList'].apply(
        lambda x: x.get('legacyProductId') if isinstance(x, dict) else None
    )
    product_group['productName'] = product_group['groupList'].apply(
        lambda x: x.get('productName') if isinstance(x, dict) else None
    )
    product_group.drop(columns='groupList', inplace=True)
    
    # 제품 그룹 테이블을 GCS에 저장
    save_to_gcs(product_group, 'product_group.csv')

    # 메인 데이터에서 그룹명만 유지
    result_list = []
    for i in range(len(meta_data)):
        value = meta_data[group_col].iloc[i]
        try:
            value = ast.literal_eval(str(value))
            group_names = [item.get('groupName') for item in value if isinstance(item, dict)]
            result_list.append(group_names)
        except:
            result_list.append(None)

    meta_data['otherOnboardInfo.duplicateNameOnboard.productGroup.groupList'] = result_list
    meta_data.drop(columns=[group_col], inplace=True)

## 9. 혜택 리스트 병합

In [ ]:
# 혜택 관련 필드 3종 병합
print("Merging benefit lists...")

benefit_cols = [
    'productBenefitConditions.allOfferBenefits.benefitInfo',
    'productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList',
    'productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList'
]

result_list = []
for i in range(len(meta_data)):
    merged_benefits = []
    
    for col in benefit_cols:
        if col in meta_data.columns:
            val = meta_data[col].iloc[i]
            try:
                val = ast.literal_eval(str(val)) if pd.notna(val) else []
                if not isinstance(val, list):
                    val = [val] if val else []
                merged_benefits.extend(val)
            except:
                pass
    
    # 중복 제거
    seen = set()
    unique_benefits = []
    for benefit in merged_benefits:
        if isinstance(benefit, dict):
            benefit_tuple = tuple(sorted(benefit.items()))
            if benefit_tuple not in seen:
                seen.add(benefit_tuple)
                unique_benefits.append(benefit)
    
    result_list.append(unique_benefits)

meta_data['productBenefitConditions.allBenefitList'] = result_list

# 원래 컬럼들 삭제
existing_benefit_cols = [col for col in benefit_cols if col in meta_data.columns]
meta_data.drop(columns=existing_benefit_cols, inplace=True)

## 10. 상품 관계 정제

In [ ]:
# 상품 간 관계 정제 (groupList와 productList 병합)
print("Processing product relations...")

relation_mappings = {
    'signupConcurrentTermination': [
        'productRelation.signupConcurrentTermination.productInformation.groupList',
        'productRelation.signupConcurrentTermination.productInformation.productList'
    ],
    'signupPreTermination': [
        'productRelation.signupPreTermination.productInformation.groupList',
        'productRelation.signupPreTermination.productInformation.productList'
    ],
    'terminationConcurrentTermination': [
        'productRelation.terminationConcurrentTermination.productInformation.groupList',
        'productRelation.terminationConcurrentTermination.productInformation.productList'
    ],
    'terminationPreTermination': [
        'productRelation.terminationPreTermination.productInformation.groupList',
        'productRelation.terminationPreTermination.productInformation.productList'
    ]
}

for relation_type, (group_col, product_col) in relation_mappings.items():
    if group_col in meta_data.columns and product_col in meta_data.columns:
        # NaN 값 채우기
        meta_data[group_col] = meta_data[group_col].fillna('[]')
        meta_data[product_col] = meta_data[product_col].fillna('[]')

        # 리스트로 변환
        meta_data[group_col] = meta_data[group_col].apply(ast.literal_eval)
        meta_data[product_col] = meta_data[product_col].apply(ast.literal_eval)

        # 리스트 병합
        merged_col = f'productRelation.{relation_type}.productList'
        meta_data[merged_col] = merge_product_lists(
            meta_data, group_col, product_col
        )

# 원래 컬럼들 삭제
drop_cols = [col for relation_cols in relation_mappings.values() for col in relation_cols 
            if col in meta_data.columns]
meta_data.drop(columns=drop_cols, inplace=True)

In [ ]:
# Step 3 저장
save_to_gcs(meta_data, 'step_3.csv')
print("Step 3 completed")

## 11. 관계 데이터베이스 생성

In [ ]:
# 혜택, 옵션 데이터, 상품 관계를 정제하여 관계 DB 생성
print("Building relation database...")

relation_db = pd.DataFrame()

# 혜택 관계 처리
benefit_col = 'productBenefitConditions.allBenefitList'
if benefit_col in meta_data.columns:
    tmp_df = meta_data[['pmProductID', benefit_col]].copy()
    tmp_df = tmp_df.explode(column=benefit_col)
    tmp_df = tmp_df.reset_index(drop=True)
    tmp_df = tmp_df[~tmp_df[benefit_col].isnull()]
    tmp_df = tmp_df.reset_index(drop=True)

    if len(tmp_df) > 0:
        tmp_df['productId'] = tmp_df[benefit_col].apply(
            lambda x: list(x.values())[0] if isinstance(x, dict) and len(x) > 0 else ''
        )
        tmp_df['productName'] = tmp_df[benefit_col].apply(
            lambda x: list(x.values())[1] if isinstance(x, dict) and len(x) > 1 else ''
        )
        tmp_df['type'] = 'productBenefitConditions.allBenefitList'
        tmp_df.drop(columns=[benefit_col], inplace=True)

        relation_db = pd.concat([relation_db, tmp_df])

# 옵션 데이터 관계 처리
option_col = 'optionData.dataOptionProvidingMethod'
if option_col in meta_data.columns:
    tmp_df = meta_data[['pmProductID', option_col]].copy()
    tmp_df[option_col] = tmp_df[option_col].apply(safe_literal_eval)
    tmp_df = tmp_df.explode(column=option_col)
    tmp_df = tmp_df.reset_index(drop=True)
    tmp_df = tmp_df[~tmp_df[option_col].isnull()]

    if len(tmp_df) > 0:
        tmp_df['productId'] = tmp_df[option_col].apply(
            lambda x: x.get('productId') if isinstance(x, dict) else ''
        )
        tmp_df['productName'] = tmp_df[option_col].apply(
            lambda x: x.get('productName') if isinstance(x, dict) else ''
        )
        
        # productId와 productName이 모두 null/empty인 행 제거
        tmp_df = tmp_df[~(tmp_df['productId'].isnull() & tmp_df['productName'].isnull())]
        tmp_df = tmp_df.reset_index(drop=True)
        
        tmp_df['type'] = 'optionData.dataOptionProvidingMethod'
        tmp_df.drop(columns=[option_col], inplace=True)
        
        relation_db = pd.concat([relation_db, tmp_df])
        relation_db = relation_db.reset_index(drop=True)

# 상품 관계 처리
relation_types = [
    'productRelation.signupConcurrentTermination.productList',
    'productRelation.signupPreTermination.productList',
    'productRelation.terminationConcurrentTermination.productList',
    'productRelation.terminationPreTermination.productList'
]

for relation_type in relation_types:
    if relation_type in meta_data.columns:
        relation_df = process_relation_data(meta_data, relation_type, relation_type)
        if len(relation_df) > 0:
            relation_db = pd.concat([relation_db, relation_df], ignore_index=True)
            
# 관계 데이터베이스 정리
if len(relation_db) > 0:
    # 누락값 채우기
    relation_db['productId'] = relation_db['productId'].fillna('')

    # concurrent termination 우선하여 중복 제거
    relation_db = remove_duplicates_keep_concurrent(relation_db)

    # 컬럼 순서 정렬
    column_order = ['relationID', 'pmProductID', 'productId', 'productName', 'type']
    existing_cols = [col for col in column_order if col in relation_db.columns]
    relation_db = relation_db[existing_cols]

save_to_gcs(relation_db, 'relation_db.csv')
print(f"Relation database created with {len(relation_db)} records")

## 12. 최종 테이블 생성

In [ ]:
# 설정에 따라 최종 테이블들 생성
print("Creating output tables...")

for table_name, config in TABLE_COLUMNS.items():
    print(f"Creating {table_name} table...")
    table_df = create_table_from_config(meta_data, table_name, config)
    
    if not table_df.empty:
        filename = f"{table_name}.csv"
        save_to_gcs(table_df, filename)
        print(f"Saved {filename} with {len(table_df)} records")
    else:
        print(f"Warning: {table_name} table is empty")

print("All tables created successfully")

In [ ]:
print("Preprocessing pipeline completed successfully!")
print(f"Results saved to: gs://{gcs_bucket_name}/{output_prefix}")